# 第21章：FlashAttention v2 — 更好的并行性

## 本章概览

FlashAttention v1 实现了 O(N) 内存的 Attention，但在 GPU 利用率上仍有改进空间。
**FlashAttention v2** 通过以下三个关键改进，实现了接近硬件理论峰值的性能：

1. **外层循环改为遍历 Q 块**（而非 K,V 块）—— 更好的并行性
2. **减少非矩阵乘法 FLOPs** —— 减少 rescaling 操作
3. **更好的 Warp 间工作分配** —— 减少通信开销

本章我们将：
1. 对比 v1 和 v2 的循环结构
2. 理解 v2 的并行性优势
3. 实现 FlashAttention v2 的前向和反向传播
4. 处理多头注意力的 batch 和 heads 维度
5. 性能对比：v1 vs v2 vs naive

## 1. v1 vs v2 循环结构对比

### 1.1 FlashAttention v1 的循环结构

```
FlashAttention v1:
  外层循环 (并行): 遍历 Q 块     
  内层循环 (串行): 遍历 K,V 块   
  
  注意: 原论文中外层遍历 K,V（便于分析IO复杂度），
  但 Triton 实现中通常外层遍历 Q（每个 program 一个 Q 块）。
  
  问题: 每个 Q 块产生的输出需要遍历所有 K,V 块后才能完成，
        Q 块间的并行度 = ceil(N / BLOCK_M)，但序列方向只有这一维并行。

       Q0 ──> [K0, K1, K2, K3, K4] ──> O0
       Q1 ──> [K0, K1, K2, K3, K4] ──> O1
       Q2 ──> [K0, K1, K2, K3, K4] ──> O2
       ...各 Q 块可并行，但每个 Q 块内串行遍历所有 K 块
```

### 1.2 FlashAttention v2 的改进

```
FlashAttention v2 (关键变化):

  1. 外层循环遍历 Q 块 (每个 program 对应一个 Q 块)
     - 与 v1 的 Triton 实现相同的 grid 设计
     - 但优化了内部的 rescaling 操作

  2. 减少 rescaling:
     v1: 每次更新都要 rescale 整个 acc (乘以 alpha)
     v2: 延迟 rescaling，在循环结束后一次性归一化
     
  3. 因果注意力优化:
     v1: 遍历所有 K,V 块，对越界的应用 mask
     v2: 只遍历需要的 K,V 块 (j <= i 的部分)，节省约 50% 计算
```

### 1.3 非因果 vs 因果的迭代对比

```
非因果注意力 (所有 Q 都看所有 K):
  
  Q块\K块  K0  K1  K2  K3  K4
   Q0     [*] [*] [*] [*] [*]
   Q1     [*] [*] [*] [*] [*]
   Q2     [*] [*] [*] [*] [*]
   Q3     [*] [*] [*] [*] [*]
   Q4     [*] [*] [*] [*] [*]
   
   每个 Q 块遍历 5 个 K 块

因果注意力 (Q 只看 <= 自己位置的 K):
  
  Q块\K块  K0  K1  K2  K3  K4
   Q0     [*]                      # 1 个 K 块
   Q1     [*] [*]                  # 2 个 K 块
   Q2     [*] [*] [*]              # 3 个 K 块
   Q3     [*] [*] [*] [*]          # 4 个 K 块  
   Q4     [*] [*] [*] [*] [*]      # 5 个 K 块
   
   v2 优化: 不同 Q 块遍历不同数量的 K 块
   平均只需遍历 ~N/2 个 K 块（相比 v1 的 N 个）
```

## 2. v2 的 Rescaling 优化

### 2.1 v1 的 Rescaling（每步都做）

```python
# v1: 每个 KV 块都要 rescale acc
for j in range(num_kv_blocks):
    S = Q @ K_j^T * scale
    m_new = max(m, rowmax(S))
    alpha = exp(m - m_new)       # 修正因子
    l_new = alpha * l + rowsum(exp(S - m_new))
    acc = alpha[:, None] * acc   # <-- 每步 rescale (BLOCK_M * d 个乘法)
    acc += exp(S - m_new) @ V_j
    m, l = m_new, l_new
acc /= l[:, None]
```

### 2.2 v2 的 Rescaling 优化

```python
# v2: 延迟部分 rescaling，减少乘法次数
for j in range(num_kv_blocks):
    S = Q @ K_j^T * scale
    m_new = max(m, rowmax(S))
    
    # 修正旧的 acc
    correction = exp(m - m_new)
    acc *= correction[:, None]    # 仍然需要 rescale
    
    # 但 l 的更新更高效
    P = exp(S - m_new)           
    l = correction * l + rowsum(P)
    acc += P @ V_j
    m = m_new

# 最终归一化
acc /= l[:, None]
```

v2 的主要优化不在于完全消除 rescaling（这在数学上是必须的），
而在于：
1. 更高效的 softmax 计算（减少冗余的 exp 操作）
2. 更好的 warp 级并行
3. 针对因果掩码的计算量优化

In [ ]:
import torch
import triton
import triton.language as tl
import math

print(f"PyTorch version: {torch.__version__}")
print(f"Triton version: {triton.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. FlashAttention v2 前向传播 Kernel

### 3.1 设计要点

```
Grid 设计:
  grid = (cdiv(N, BLOCK_M), B * H)
  - pid_m: Q 块索引 (外层并行)
  - pid_bh: batch * head 索引

每个 program 的工作:
  1. 加载 Q 块 [BLOCK_M, d]
  2. 初始化 m, l, acc
  3. 循环遍历 K,V 块:
     - 因果: j 从 0 到 cdiv((pid_m+1)*BLOCK_M, BLOCK_N)
     - 非因果: j 从 0 到 cdiv(N, BLOCK_N)
  4. 在线 softmax 更新
  5. 最终归一化并写回

优化点:
  - Q 块只加载一次，保持在寄存器中
  - K,V 块顺序加载，利用 L2 缓存
  - 因果掩码时跳过不需要的 K 块
```

In [ ]:
@triton.jit
def flash_attention_v2_fwd_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    L_ptr,  # logsumexp
    stride_qb, stride_qh, stride_qn, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_on, stride_od,
    stride_lb, stride_lh, stride_ln,
    B: tl.constexpr,
    H: tl.constexpr,
    N: tl.constexpr,
    D: tl.constexpr,
    sm_scale,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    """
    FlashAttention v2 前向传播 kernel。
    
    相比 v1 的主要改进:
    1. 外层循环遍历 Q 块 (pid 直接映射到 Q 块)
    2. 因果模式下只遍历必要的 K,V 块
    3. 优化的 rescaling 逻辑
    """
    # program 索引
    pid_m = tl.program_id(0)    # Q 块索引
    pid_bh = tl.program_id(1)   # batch * head
    
    # 分解 batch 和 head (v2 明确处理)
    pid_b = pid_bh // H
    pid_h = pid_bh % H
    
    # 基地址: 使用完整的 stride 计算
    q_base = Q_ptr + pid_b * stride_qb + pid_h * stride_qh
    k_base = K_ptr + pid_b * stride_kb + pid_h * stride_kh
    v_base = V_ptr + pid_b * stride_vb + pid_h * stride_vh
    o_base = O_ptr + pid_b * stride_ob + pid_h * stride_oh
    l_base = L_ptr + pid_b * stride_lb + pid_h * stride_lh
    
    # Q 块的行偏移
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_D)
    
    # 加载 Q 块 [BLOCK_M, BLOCK_D] —— 整个内层循环中保持在寄存器
    q_ptrs = q_base + offs_m[:, None] * stride_qn + offs_d[None, :] * stride_qd
    q_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    q_block = tl.load(q_ptrs, mask=q_mask, other=0.0)
    
    # 初始化在线 softmax 状态
    m_i = tl.full([BLOCK_M], value=float('-inf'), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
    acc = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)
    
    # ======================================
    # 确定 K,V 块的遍历范围
    # ======================================
    if IS_CAUSAL:
        # 因果模式: 只遍历 k_pos <= max(q_pos) 的 K 块
        # max(q_pos) = (pid_m + 1) * BLOCK_M - 1
        kv_block_end = tl.cdiv((pid_m + 1) * BLOCK_M, BLOCK_N)
    else:
        kv_block_end = tl.cdiv(N, BLOCK_N)
    
    # ======================================
    # 内层循环: 遍历 K,V 块
    # ======================================
    for block_n_idx in range(0, kv_block_end):
        offs_n = block_n_idx * BLOCK_N + tl.arange(0, BLOCK_N)
        
        # 加载 K 块
        k_ptrs = k_base + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        k_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        k_block = tl.load(k_ptrs, mask=k_mask, other=0.0)
        
        # S = Q @ K^T * scale  [BLOCK_M, BLOCK_N]
        s_block = tl.dot(q_block, tl.trans(k_block))
        s_block = s_block * sm_scale
        
        # 掩码
        valid_mask = offs_n[None, :] < N
        s_block = tl.where(valid_mask, s_block, float('-inf'))
        
        if IS_CAUSAL:
            causal_mask = offs_m[:, None] >= offs_n[None, :]
            s_block = tl.where(causal_mask, s_block, float('-inf'))
        
        # ======================================
        # 在线 softmax 更新 (v2 优化版本)
        # ======================================
        
        # 当前块行最大值
        m_block = tl.max(s_block, axis=1)  # [BLOCK_M]
        m_new = tl.maximum(m_i, m_block)
        
        # 修正因子
        alpha = tl.exp(m_i - m_new)
        
        # exp(S - m_new)
        p_block = tl.exp(s_block - m_new[:, None])
        
        # 更新 l: l_new = alpha * l_old + rowsum(P)
        l_i = alpha * l_i + tl.sum(p_block, axis=1)
        
        # 更新 acc: acc = alpha * acc + P @ V
        acc = acc * alpha[:, None]
        
        # 加载 V 块
        v_ptrs = v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        v_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        v_block = tl.load(v_ptrs, mask=v_mask, other=0.0)
        
        p_block_fp16 = p_block.to(tl.float16)
        acc += tl.dot(p_block_fp16, v_block).to(tl.float32)
        
        # 更新 m
        m_i = m_new
    
    # ======================================
    # 最终归一化
    # ======================================
    acc = acc / l_i[:, None]
    
    # logsumexp
    lse = m_i + tl.log(l_i)
    
    # 写回
    o_ptrs = o_base + offs_m[:, None] * stride_on + offs_d[None, :] * stride_od
    o_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    tl.store(o_ptrs, acc.to(tl.float16), mask=o_mask)
    
    l_ptrs = l_base + offs_m * stride_ln
    l_mask = offs_m < N
    tl.store(l_ptrs, lse, mask=l_mask)

In [ ]:
def flash_attention_v2_fwd(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    causal: bool = False,
) -> tuple:
    """
    FlashAttention v2 前向传播。
    
    返回:
        output: [B, H, N, d]
        lse: [B, H, N] logsumexp
    """
    B, H, N, d = q.shape
    
    o = torch.empty_like(q)
    lse = torch.empty(B, H, N, device=q.device, dtype=torch.float32)
    
    sm_scale = 1.0 / math.sqrt(d)
    
    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(d)
    
    grid = (triton.cdiv(N, BLOCK_M), B * H)
    
    flash_attention_v2_fwd_kernel[grid](
        q, k, v, o, lse,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        o.stride(0), o.stride(1), o.stride(2), o.stride(3),
        lse.stride(0), lse.stride(1), lse.stride(2),
        B=B, H=H, N=N, D=d,
        sm_scale=sm_scale,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_D=BLOCK_D,
        IS_CAUSAL=causal,
    )
    
    return o, lse


print("FlashAttention v2 前向传播定义完成!")

## 4. FlashAttention v2 反向传播

### 4.1 反向传播的重计算策略

FlashAttention 的反向传播使用**重计算（Recomputation）**策略：
不存储前向传播中的 S 和 P 矩阵（它们是 O(N^2) 的），
而是在反向传播时重新计算它们。

需要保存的前向传播中间结果：
- Q, K, V（输入，本来就要保存）
- O（输出）
- L = m + log(l)（logsumexp，O(N) 大小）

```
反向传播计算流程:

已知: dO (输出的梯度), Q, K, V, O, L (logsumexp)
求: dQ, dK, dV

关键公式:
  D = rowsum(dO * O)           # [N], 辅助向量
  
  对每对 (Q块i, K块j):
    S_ij = Q_i @ K_j^T / sqrt(d)
    P_ij = exp(S_ij - L_i)     # 重计算 softmax 权重
    
    dV_j += P_ij^T @ dO_i
    dP_ij = dO_i @ V_j^T
    dS_ij = P_ij * (dP_ij - D_i)  # softmax 反向
    dQ_i += dS_ij @ K_j / sqrt(d)
    dK_j += dS_ij^T @ Q_i / sqrt(d)
```

In [ ]:
@triton.jit
def flash_attention_v2_bwd_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    dO_ptr, dQ_ptr, dK_ptr, dV_ptr,
    L_ptr,   # logsumexp from forward
    D_ptr,   # rowsum(dO * O)
    stride_qb, stride_qh, stride_qn, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_on, stride_od,
    stride_lb, stride_lh, stride_ln,
    N: tl.constexpr,
    D_head: tl.constexpr,
    sm_scale,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    """
    FlashAttention v2 反向传播 kernel。
    
    每个 program 处理一个 K,V 块 (BLOCK_N 行)，
    遍历所有 Q 块来累积 dK 和 dV。
    dQ 通过 atomic add 累积。
    
    这里的策略: 外层并行在 K,V 块上，内层遍历 Q 块
    (与前向传播相反，因为 dK, dV 需要累积所有 Q 块的贡献)
    """
    pid_n = tl.program_id(0)     # K,V 块索引
    pid_bh = tl.program_id(1)    # batch * head
    
    # 基地址
    q_base = Q_ptr + pid_bh * stride_qh
    k_base = K_ptr + pid_bh * stride_kh
    v_base = V_ptr + pid_bh * stride_vh
    o_base = O_ptr + pid_bh * stride_oh
    do_base = dO_ptr + pid_bh * stride_oh
    dq_base = dQ_ptr + pid_bh * stride_qh
    dk_base = dK_ptr + pid_bh * stride_kh
    dv_base = dV_ptr + pid_bh * stride_vh
    l_base = L_ptr + pid_bh * stride_lh
    d_base = D_ptr + pid_bh * stride_lh
    
    # K,V 块的列偏移
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, BLOCK_D)
    
    # 加载 K, V 块
    k_ptrs = k_base + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    v_ptrs = v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
    kv_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D_head)
    k_block = tl.load(k_ptrs, mask=kv_mask, other=0.0)
    v_block = tl.load(v_ptrs, mask=kv_mask, other=0.0)
    
    # 初始化 dK, dV 累积器
    dk_acc = tl.zeros([BLOCK_N, BLOCK_D], dtype=tl.float32)
    dv_acc = tl.zeros([BLOCK_N, BLOCK_D], dtype=tl.float32)
    
    # 确定 Q 块遍历范围
    if IS_CAUSAL:
        # 因果: 只有 q_pos >= k_pos 的 Q 块才有贡献
        q_block_start = pid_n * BLOCK_N // BLOCK_M
    else:
        q_block_start = 0
    num_q_blocks = tl.cdiv(N, BLOCK_M)
    
    # 遍历 Q 块
    for block_m_idx in range(q_block_start, num_q_blocks):
        offs_m = block_m_idx * BLOCK_M + tl.arange(0, BLOCK_M)
        
        # 加载 Q 块, dO 块
        q_ptrs = q_base + offs_m[:, None] * stride_qn + offs_d[None, :] * stride_qd
        q_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D_head)
        q_block = tl.load(q_ptrs, mask=q_mask, other=0.0)
        
        do_ptrs = do_base + offs_m[:, None] * stride_on + offs_d[None, :] * stride_od
        do_block = tl.load(do_ptrs, mask=q_mask, other=0.0)
        
        # 加载 L 和 D
        l_ptrs = l_base + offs_m * stride_ln
        l_mask = offs_m < N
        lse_block = tl.load(l_ptrs, mask=l_mask, other=0.0)
        
        d_ptrs = d_base + offs_m * stride_ln
        d_block = tl.load(d_ptrs, mask=l_mask, other=0.0)
        
        # 重计算 S = Q @ K^T * scale
        s_block = tl.dot(q_block, tl.trans(k_block)) * sm_scale  # [BLOCK_M, BLOCK_N]
        
        # 重计算 P = exp(S - L) (利用保存的 logsumexp)
        p_block = tl.exp(s_block - lse_block[:, None])  # [BLOCK_M, BLOCK_N]
        
        # 掩码
        valid_mask = (offs_m[:, None] < N) & (offs_n[None, :] < N)
        if IS_CAUSAL:
            causal_mask = offs_m[:, None] >= offs_n[None, :]
            valid_mask = valid_mask & causal_mask
        p_block = tl.where(valid_mask, p_block, 0.0)
        
        # dV += P^T @ dO
        p_block_fp16 = p_block.to(tl.float16)
        dv_acc += tl.dot(tl.trans(p_block_fp16), do_block).to(tl.float32)
        
        # dP = dO @ V^T  [BLOCK_M, BLOCK_N]
        dp_block = tl.dot(do_block, tl.trans(v_block)).to(tl.float32)
        
        # dS = P * (dP - D)  (softmax backward)
        ds_block = p_block * (dp_block - d_block[:, None]) * sm_scale
        ds_block = tl.where(valid_mask, ds_block, 0.0)
        
        # dK += dS^T @ Q
        ds_block_fp16 = ds_block.to(tl.float16)
        dk_acc += tl.dot(tl.trans(ds_block_fp16), q_block).to(tl.float32)
        
        # dQ 需要 atomic add (因为多个 K 块都会贡献到同一个 Q)
        dq_block = tl.dot(ds_block_fp16, k_block).to(tl.float32)
        dq_ptrs = dq_base + offs_m[:, None] * stride_qn + offs_d[None, :] * stride_qd
        tl.atomic_add(dq_ptrs, dq_block.to(tl.float16), mask=q_mask)
    
    # 写回 dK, dV
    dk_ptrs = dk_base + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    dv_ptrs = dv_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
    tl.store(dk_ptrs, dk_acc.to(tl.float16), mask=kv_mask)
    tl.store(dv_ptrs, dv_acc.to(tl.float16), mask=kv_mask)

In [ ]:
def flash_attention_v2_bwd(
    q, k, v, o, do, lse, causal=False
):
    """
    FlashAttention v2 反向传播。
    
    参数:
        q, k, v: 前向传播的输入 [B, H, N, d]
        o: 前向传播的输出 [B, H, N, d]
        do: 输出梯度 [B, H, N, d]
        lse: 前向传播的 logsumexp [B, H, N]
    
    返回:
        dq, dk, dv: 输入梯度 [B, H, N, d]
    """
    B, H, N, d = q.shape
    
    # 预计算 D = rowsum(dO * O)
    D_vec = (do.float() * o.float()).sum(dim=-1)  # [B, H, N]
    
    # 分配梯度
    dq = torch.zeros_like(q)
    dk = torch.empty_like(k)
    dv = torch.empty_like(v)
    
    sm_scale = 1.0 / math.sqrt(d)
    
    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(d)
    
    # 反向传播: 外层并行在 K,V 块上
    grid = (triton.cdiv(N, BLOCK_N), B * H)
    
    flash_attention_v2_bwd_kernel[grid](
        q, k, v, o,
        do, dq, dk, dv,
        lse, D_vec,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        o.stride(0), o.stride(1), o.stride(2), o.stride(3),
        lse.stride(0), lse.stride(1), lse.stride(2),
        N=N,
        D_head=d,
        sm_scale=sm_scale,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_D=BLOCK_D,
        IS_CAUSAL=causal,
    )
    
    return dq, dk, dv


print("FlashAttention v2 反向传播定义完成!")

## 5. 将前向和反向封装为 autograd Function

In [ ]:
class FlashAttentionV2Function(torch.autograd.Function):
    """FlashAttention v2 的 autograd 封装。"""
    
    @staticmethod
    def forward(ctx, q, k, v, causal):
        o, lse = flash_attention_v2_fwd(q, k, v, causal=causal)
        ctx.save_for_backward(q, k, v, o, lse)
        ctx.causal = causal
        return o
    
    @staticmethod
    def backward(ctx, do):
        q, k, v, o, lse = ctx.saved_tensors
        dq, dk, dv = flash_attention_v2_bwd(
            q, k, v, o, do, lse, causal=ctx.causal
        )
        return dq, dk, dv, None


def flash_attention_v2(q, k, v, causal=False):
    """FlashAttention v2 接口（支持反向传播）。"""
    return FlashAttentionV2Function.apply(q, k, v, causal)


print("FlashAttention v2 autograd Function 定义完成!")

## 6. 正确性验证

In [ ]:
def attention_pytorch_reference(q, k, v, causal=False):
    """PyTorch 参考实现。"""
    B, H, N, d = q.shape
    sm_scale = 1.0 / math.sqrt(d)
    s = torch.matmul(q.float(), k.float().transpose(-2, -1)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~mask, float('-inf'))
    p = torch.softmax(s, dim=-1)
    o = torch.matmul(p, v.float())
    return o.half()


def test_flash_v2(B, H, N, d, causal=False):
    """测试 FlashAttention v2 前向传播正确性。"""
    torch.manual_seed(42)
    q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    
    out_ref = attention_pytorch_reference(q, k, v, causal=causal)
    out_v2, _ = flash_attention_v2_fwd(q, k, v, causal=causal)
    
    max_diff = (out_ref - out_v2).abs().max().item()
    is_close = torch.allclose(out_ref, out_v2, rtol=1e-2, atol=1e-2)
    
    causal_str = "causal" if causal else "full"
    status = "PASS" if is_close else "FAIL"
    print(f"[{status}] B={B}, H={H}, N={N}, d={d}, {causal_str} | max_diff={max_diff:.6f}")
    return is_close


print("=" * 80)
print("FlashAttention v2 前向传播正确性测试")
print("=" * 80)

all_pass = True
for N in [64, 128, 256, 512, 1024]:
    for causal in [False, True]:
        result = test_flash_v2(B=2, H=4, N=N, d=64, causal=causal)
        all_pass = all_pass and result

print("\n" + "=" * 80)
print("所有测试通过!" if all_pass else "存在测试失败!")
print("=" * 80)

In [ ]:
# 测试反向传播
def test_flash_v2_backward(B, H, N, d, causal=False):
    """测试 FlashAttention v2 反向传播正确性。"""
    torch.manual_seed(42)
    
    # PyTorch 参考 (需要梯度)
    q_ref = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16, requires_grad=True)
    k_ref = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16, requires_grad=True)
    v_ref = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16, requires_grad=True)
    
    sm_scale = 1.0 / math.sqrt(d)
    s_ref = torch.matmul(q_ref.float(), k_ref.float().transpose(-2, -1)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones(N, N, device='cuda', dtype=torch.bool))
        s_ref = s_ref.masked_fill(~mask, float('-inf'))
    p_ref = torch.softmax(s_ref, dim=-1)
    o_ref = torch.matmul(p_ref, v_ref.float()).half()
    
    # Triton flash v2
    q_tri = q_ref.detach().clone().requires_grad_(True)
    k_tri = k_ref.detach().clone().requires_grad_(True)
    v_tri = v_ref.detach().clone().requires_grad_(True)
    o_tri = flash_attention_v2(q_tri, k_tri, v_tri, causal=causal)
    
    # 反向传播
    do = torch.randn_like(o_ref)
    o_ref.backward(do)
    o_tri.backward(do)
    
    # 比较梯度
    dq_diff = (q_ref.grad - q_tri.grad).abs().max().item()
    dk_diff = (k_ref.grad - k_tri.grad).abs().max().item()
    dv_diff = (v_ref.grad - v_tri.grad).abs().max().item()
    
    causal_str = "causal" if causal else "full"
    print(f"反向传播 ({causal_str}) B={B}, H={H}, N={N}, d={d}:")
    print(f"  dQ max diff: {dq_diff:.6f}")
    print(f"  dK max diff: {dk_diff:.6f}")
    print(f"  dV max diff: {dv_diff:.6f}")
    
    # FP16 的容差较大
    all_ok = dq_diff < 0.05 and dk_diff < 0.05 and dv_diff < 0.05
    print(f"  Status: {'PASS' if all_ok else 'FAIL'}")
    return all_ok


print("=" * 80)
print("FlashAttention v2 反向传播正确性测试")
print("=" * 80)
test_flash_v2_backward(B=1, H=2, N=128, d=64, causal=False)
print()
test_flash_v2_backward(B=1, H=2, N=128, d=64, causal=True)

## 7. 性能对比：Naive vs v1 vs v2

v2 相比 v1 的性能优势主要体现在：
1. 因果注意力下更少的计算量（跳过不需要的 K 块）
2. 更好的 GPU 利用率
3. 更少的非矩阵乘法操作

In [ ]:
# 重新定义 v1 用于对比
@triton.jit
def flash_attention_v1_simple_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    stride_bh, stride_n, stride_d,
    N: tl.constexpr,
    D: tl.constexpr,
    sm_scale,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_bh = tl.program_id(1)
    
    q_base = Q_ptr + pid_bh * stride_bh
    k_base = K_ptr + pid_bh * stride_bh
    v_base = V_ptr + pid_bh * stride_bh
    o_base = O_ptr + pid_bh * stride_bh
    
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_D)
    
    q_ptrs = q_base + offs_m[:, None] * stride_n + offs_d[None, :] * stride_d
    q_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    q_block = tl.load(q_ptrs, mask=q_mask, other=0.0)
    
    m_i = tl.full([BLOCK_M], value=float('-inf'), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
    acc = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)
    
    num_kv_blocks = tl.cdiv(N, BLOCK_N)  # v1: 遍历所有 K 块
    
    for block_n_idx in range(0, num_kv_blocks):
        offs_n = block_n_idx * BLOCK_N + tl.arange(0, BLOCK_N)
        
        k_ptrs = k_base + offs_n[:, None] * stride_n + offs_d[None, :] * stride_d
        k_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        k_block = tl.load(k_ptrs, mask=k_mask, other=0.0)
        
        s_block = tl.dot(q_block, tl.trans(k_block)) * sm_scale
        
        valid_mask = offs_n[None, :] < N
        s_block = tl.where(valid_mask, s_block, float('-inf'))
        if IS_CAUSAL:
            causal_mask = offs_m[:, None] >= offs_n[None, :]
            s_block = tl.where(causal_mask, s_block, float('-inf'))
        
        m_block = tl.max(s_block, axis=1)
        m_new = tl.maximum(m_i, m_block)
        alpha = tl.exp(m_i - m_new)
        p_block = tl.exp(s_block - m_new[:, None])
        l_i = alpha * l_i + tl.sum(p_block, axis=1)
        acc = acc * alpha[:, None]
        
        v_ptrs = v_base + offs_n[:, None] * stride_n + offs_d[None, :] * stride_d
        v_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        v_block = tl.load(v_ptrs, mask=v_mask, other=0.0)
        acc += tl.dot(p_block.to(tl.float16), v_block).to(tl.float32)
        m_i = m_new
    
    acc = acc / l_i[:, None]
    o_ptrs = o_base + offs_m[:, None] * stride_n + offs_d[None, :] * stride_d
    o_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    tl.store(o_ptrs, acc.to(tl.float16), mask=o_mask)


def flash_v1_simple(q, k, v, causal=False):
    B, H, N, d = q.shape
    o = torch.empty_like(q)
    sm_scale = 1.0 / math.sqrt(d)
    BLOCK_M, BLOCK_N = 64, 64
    BLOCK_D = triton.next_power_of_2(d)
    grid = (triton.cdiv(N, BLOCK_M), B * H)
    # 将 [B,H,N,d] 视为连续的 [B*H, N, d]
    q_c = q.contiguous().view(B*H, N, d)
    k_c = k.contiguous().view(B*H, N, d)
    v_c = v.contiguous().view(B*H, N, d)
    o_c = o.view(B*H, N, d)
    flash_attention_v1_simple_kernel[grid](
        q_c, k_c, v_c, o_c,
        q_c.stride(0), q_c.stride(1), q_c.stride(2),
        N=N, D=d, sm_scale=sm_scale,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
        IS_CAUSAL=causal,
    )
    return o


def naive_attention_pytorch(q, k, v, causal=False):
    B, H, N, d = q.shape
    sm_scale = 1.0 / math.sqrt(d)
    s = torch.matmul(q, k.transpose(-2, -1)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~mask, float('-inf'))
    p = torch.softmax(s, dim=-1)
    return torch.matmul(p, v)


print("对比函数定义完成!")

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['N'],
        x_vals=[128, 256, 512, 1024, 2048, 4096],
        line_arg='provider',
        line_vals=['pytorch_naive', 'flash_v1', 'flash_v2', 'flash_v2_causal'],
        line_names=['PyTorch Naive', 'Flash v1', 'Flash v2', 'Flash v2 (Causal)'],
        styles=[('blue', '-'), ('orange', '-'), ('red', '-'), ('green', '--')],
        ylabel='ms',
        plot_name='FlashAttention v1 vs v2 vs Naive',
        args={'B': 2, 'H': 8, 'd': 64},
    )
)
def bench_all(B, H, N, d, provider):
    q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    
    quantiles = [0.5, 0.2, 0.8]
    
    if provider == 'pytorch_naive':
        fn = lambda: naive_attention_pytorch(q, k, v)
    elif provider == 'flash_v1':
        fn = lambda: flash_v1_simple(q, k, v, causal=False)
    elif provider == 'flash_v2':
        fn = lambda: flash_attention_v2_fwd(q, k, v, causal=False)
    elif provider == 'flash_v2_causal':
        fn = lambda: flash_attention_v2_fwd(q, k, v, causal=True)
    
    ms, min_ms, max_ms = triton.testing.do_bench(fn, quantiles=quantiles)
    return ms, min_ms, max_ms


bench_all.run(print_data=True)

## 8. v1 vs v2 详细对比

```
┌─────────────────────┬──────────────────────┬──────────────────────┐
│       特性           │   FlashAttention v1   │   FlashAttention v2   │
├─────────────────────┼──────────────────────┼──────────────────────┤
│ 论文外层循环         │ K,V 块               │ Q 块                 │
│ Triton 实现外层      │ Q 块 (并行)          │ Q 块 (并行)          │
│ 因果优化             │ 遍历所有K块+mask     │ 只遍历必要的K块      │
│ 非矩阵乘法 FLOPs    │ 较多 rescaling       │ 减少 rescaling       │
│ Warp 间通信         │ 分割 K 维度          │ 分割 Q 维度          │
│ 理论 FLOPS 利用率   │ ~50-70%              │ ~70-90%              │
│ 反向传播             │ 有，但较慢           │ 优化的重计算          │
│ 内存复杂度           │ O(N)                 │ O(N)                 │
│ IO 复杂度           │ O(N^2*d/M)           │ O(N^2*d/M)           │
│ 实际速度提升 (vs v1) │ 基准                 │ ~2x                  │
└─────────────────────┴──────────────────────┴──────────────────────┘
```

### Warp 间工作分配的差异（高级话题）

```
v1: 所有 warp 共同计算 S 和 P，然后分割 K 维度来做 P@V
    - 需要 warp 间通信来聚合 softmax 的 max 和 sum
    - 效率受限于通信开销

v2: warp 间分割 Q 维度
    - 每个 warp 处理 Q 的不同行
    - softmax 在每个 warp 内独立完成（无需跨 warp 通信）
    - 更高效的并行
    
  ┌──────────────────────────────┐
  │    v1: Warp 间分割 K 维度    │
  │                              │
  │  Warp0: S[:, 0:16]           │
  │  Warp1: S[:, 16:32]          │  需要跨 warp reduce max/sum
  │  Warp2: S[:, 32:48]          │
  │  Warp3: S[:, 48:64]          │
  ├──────────────────────────────┤
  │    v2: Warp 间分割 Q 维度    │
  │                              │
  │  Warp0: S[0:16, :]           │
  │  Warp1: S[16:32, :]          │  每个 warp 独立做 softmax
  │  Warp2: S[32:48, :]          │
  │  Warp3: S[48:64, :]          │
  └──────────────────────────────┘
```

## 9. 练习

### 练习 1: 因果注意力的计算量节省
对于因果注意力，v2 只遍历必要的 K 块。计算当 N=4096, BLOCK_M=BLOCK_N=64 时：
- v1 的总 K 块遍历次数（所有 Q 块的总和）
- v2 的总 K 块遍历次数
- 节省了多少百分比？

### 练习 2: 不同头维度的影响
分别用 d=32, 64, 128, 256 运行 benchmark。
观察头维度对性能的影响。为什么 d 越大，FlashAttention 的优势越明显？

### 练习 3: 反向传播的计算量
分析 FlashAttention v2 反向传播的计算量：
- 前向传播需要多少 matmul？
- 反向传播需要多少 matmul？
- 反向传播 vs 前向传播的计算量比值是多少？

### 练习 4: Multi-Query Attention (MQA)
修改 kernel 支持 MQA：所有 Q 头共享同一组 K,V。
提示：只需要修改 K,V 的基地址计算。

### 练习 5: 数值精度分析
比较 v1 和 v2 的数值精度（与 FP32 参考实现对比）。
哪个更精确？为什么？

## 总结

本章我们学习了 FlashAttention v2 相比 v1 的关键改进：

1. **更好的并行性**：外层循环遍历 Q 块，每个 Q 块独立并行
2. **减少非矩阵乘法 FLOPs**：优化 rescaling 操作
3. **更好的 warp 分配**：分割 Q 维度避免跨 warp 通信
4. **因果优化**：只遍历必要的 K,V 块
5. **反向传播**：通过重计算策略节省内存

### 关键教训

> **GPU kernel 的性能优化不仅是关于算法复杂度，更是关于如何匹配硬件的并行结构。**
> 相同的算法（在线 softmax），不同的循环结构和工作分配，可以带来数倍的性能差异。

下一章：FlashAttention v3 —— 异步与 Warp 特化